# Modelado No Supervisado (Clustering)

Implementacion de K-Means para identificar segmentos ocultos dentro del dominio hotelero.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

BASE = Path(r'C:/ProyectoBD')
DATA_DIR = BASE / 'PREPROCESAMIENTO_LIMPIEZA_DATOS'
prepared = pd.read_csv(DATA_DIR / '8_dataset_preparado_ml.csv')
clean = pd.read_csv(DATA_DIR / '6_dataset_limpio.csv')
prepared.shape, clean.shape

## Seleccion de variables

Se excluyen variables de resultado o estado posterior (`is_canceled`, `reservation_status`, `reservation_status_date`) para evitar que el clustering dependa de etiquetas directas.

In [ ]:
excluded = ['is_canceled', 'reservation_status', 'reservation_status_date']
features = [c for c in prepared.columns if c not in excluded]
X = prepared[features].replace([np.inf, -np.inf], np.nan).fillna(0)
len(features), features[:10]

## Metodo del Codo y Coeficiente de Silueta

Se prueban diferentes cantidades de clusters para comparar inercia y coeficiente de silueta.

In [ ]:
k_values = range(2, 9)
inertias = []
silhouettes = []
X_sample = X.sample(n=min(10000, len(X)), random_state=42)

for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=20, max_iter=300)
    labels = model.fit_predict(X)
    inertias.append(model.inertia_)
    silhouettes.append(silhouette_score(X_sample, model.predict(X_sample)))

metrics = pd.DataFrame({'k': list(k_values), 'inercia': inertias, 'coeficiente_silueta': silhouettes})
metrics

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].plot(metrics['k'], metrics['inercia'], marker='o')
ax[0].set_title('Metodo del Codo')
ax[0].set_xlabel('k')
ax[0].set_ylabel('Inercia')
ax[0].grid(alpha=0.3)

best_k = int(metrics.loc[metrics['coeficiente_silueta'].idxmax(), 'k'])
ax[1].plot(metrics['k'], metrics['coeficiente_silueta'], marker='o', color='green')
ax[1].axvline(best_k, linestyle='--', color='red', label=f'k optimo = {best_k}')
ax[1].set_title('Coeficiente de Silueta')
ax[1].set_xlabel('k')
ax[1].set_ylabel('Silueta')
ax[1].legend()
ax[1].grid(alpha=0.3)
plt.show()

## Entrenamiento final e interpretacion

Se entrena K-Means con el numero optimo de clusters segun la silueta y se perfilan los grupos usando el dataset limpio para mantener variables interpretables.

In [ ]:
final_model = KMeans(n_clusters=best_k, random_state=42, n_init=20, max_iter=300)
clusters = final_model.fit_predict(X)
analysis = clean.copy()
analysis['cluster'] = clusters
analysis['total_noches'] = analysis['stays_in_weekend_nights'] + analysis['stays_in_week_nights']
analysis['total_huespedes'] = analysis['adults'] + analysis['children'] + analysis['babies']

summary = analysis.groupby('cluster').agg(
    total_registros=('cluster', 'size'),
    porcentaje_cancelacion=('is_canceled', 'mean'),
    lead_time_promedio=('lead_time', 'mean'),
    adr_promedio=('adr', 'mean'),
    noches_promedio=('total_noches', 'mean'),
    huespedes_promedio=('total_huespedes', 'mean'),
    solicitudes_promedio=('total_of_special_requests', 'mean')
).reset_index()
summary['porcentaje_cancelacion'] *= 100
summary['porcentaje_registros'] = summary['total_registros'] / len(analysis) * 100
summary.round(2)

In [ ]:
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_sample)
sample_labels = final_model.predict(X_sample)
plt.figure(figsize=(8, 6))
plt.scatter(coords[:, 0], coords[:, 1], c=sample_labels, cmap='tab10', s=10, alpha=0.65)
plt.title('Visualizacion de clusters con PCA')
plt.xlabel('Componente principal 1')
plt.ylabel('Componente principal 2')
plt.colorbar(label='Cluster')
plt.show()

## Resultado

El numero optimo seleccionado por el coeficiente de silueta fue **k = 2**. Las graficas y archivos de resultados se generaron en las carpetas `graficas` y `resultados`.